# Purpose
This notebook aims to explain LLM, Prompt Engineering and working with Langchain framework using LCEL . I wish you a good learning :)

## LLM
It simply means Large Langage Model. LLMs are some AI models which has been trained on a wide or various data. They are based on the paper ATTENTION IS ALL NEED. LLMs relies on probability, they try to predict the next token or next word in a sentence for instance.
As LLM , we can note:
* IBM Watson
* LLaMa of Facebook
* Chatgpt (surrely the most known)

LLMs are useful for achieving a lot of tasks. Indeed , they are able to 
* understand Human Langage through NLP(Natural Langage Processing),
* See some images through Computer vision. As a matter of fact, they detect some diseases on scanner

## Key concept of LLMs
* Token: It is a basic unit of text. You can see it as a word in certains cases
* Vector: It is a numerical representation of token .They have a direction. For instance ,the words King and Queen have near vector 'cause they presents some closest meaning
* Context Window: It is the maximum lenght of token a LLM can see at once

# Prompt Engeniering
We can see Prompt Engineering as the science of implementing some well-formed or useful input which are intended for or aimed at LLMs. Actually 
,it allows LLM to generate suitable outputs. Prompt Engineering is just a driver which leads LLM response. Since LLMs have been on range of data ,
Prompt Engineering is useful for getting suitable response

## In-Context-Learning
LLM learns from examples which have been added in prompt . We dont need to train it back or updating its weight

In [ ]:
# Langchain
For learning Langchain, we will use Ollama . It enables to work with LLM locally. 

In [1]:
from ollama import Client
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings


# Prompt template
It allows us to use templates for our prompts. Prompt template are like Blueprint for our prompr. That way ,we can reuse it . Lets see some types of prompt template

# Zero Shot template
We give any information to our LLM. 

In [2]:
zero_prompt_template = """
You are a agriculture expert. Explain this :{concept}
like you speak to a child
Answer:
"""

zero_shot_pompt = PromptTemplate(
    input_variables = ['concept'],
    template=zero_prompt_template
)

## Lets see the result of promptZeroShot
We are able to reuse this prompt template for multiple usecases. For instances, we could ask LLM to explain Photoynthesis ,crop or whatever we want about agriculture.

In [3]:
zero_shot_pompt

PromptTemplate(input_variables=['concept'], input_types={}, partial_variables={}, template='\nYou are a agriculture expert. Explain this :{concept}\nlike you speak to a child\nAnswer:\n')

# One shot Prompting
We give some tips or information to LLM. It can lead its decisions.

In [5]:
one_prompt_template = """
You are an expert . I will give you some samples of sentences and emotion it leads
Phrase 1: I dont like your behavior
It express a disapointment

Tell which sentiment is expressed in this sentence {sentence}
"""

prompt_one_shot = PromptTemplate(
    input_variables=['sentence'],
    template= one_prompt_template
)

template_one_shot

'\nYou are an expert . I will give you some samples of sentences and emotion it leads\nPhrase 1: I dont like your behavior\nit express a disapointment\n\nTell which sentiment is expressed in this sentence {sentence}\n'

# Other type of prompting template
There are others type of prompt template. Among them ,we can note:
* Few shot :Like One shot but we give more samples
* CoT (Chain of thought): We foaster the model to split a particular problems in some step-by-step or process to get a final output
* Self-Consistency

### Lets work with LLM
We can see how the LLM (model) will reply regarding to our query or prompt.
* We must start initialize our client
* Then we set our prompt by formating our prompt
*  Finally ,we ask our question to model 

In [8]:
client = Client()
prompt = zero_shot_pompt.format(concept='planting')
responseAgri = client.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])


#### Lets see the response format
With Pydantic ,we can oblige our model to reply in a specific format... We will see it in a next session

In [14]:
for _,value in enumerate(responseAgri):
    print(value)

('model', 'llama3')
('created_at', '2025-11-25T21:33:40.1723097Z')
('done', True)
('done_reason', 'stop')
('total_duration', 157686826800)
('load_duration', 13770168300)
('prompt_eval_count', 31)
('prompt_eval_duration', 7986045400)
('eval_count', 327)
('eval_duration', 134986125500)
('message', Message(role='assistant', content="Yay! Let me explain planting in a super fun way!\n\nSo, you know how we need food to eat and grow strong? Like apples and carrots and yummy bread?\n\nWell, plants need special help to grow too! And that's where planting comes in.\n\nPlanting is when we take tiny little seeds or roots of plants and put them in the ground. It's like giving a special home to these little guys so they can grow into big, strong plants!\n\nHere's how it works:\n\n1. First, we choose the right spot for our plant to grow. We need to make sure it gets enough sunlight, water, and air.\n2. Next, we take out the tiny seed or root of the plant. For example, if we're planting a carrot, we'l

In [17]:
responseAgri['message']['content']

"Yay! Let me explain planting in a super fun way!\n\nSo, you know how we need food to eat and grow strong? Like apples and carrots and yummy bread?\n\nWell, plants need special help to grow too! And that's where planting comes in.\n\nPlanting is when we take tiny little seeds or roots of plants and put them in the ground. It's like giving a special home to these little guys so they can grow into big, strong plants!\n\nHere's how it works:\n\n1. First, we choose the right spot for our plant to grow. We need to make sure it gets enough sunlight, water, and air.\n2. Next, we take out the tiny seed or root of the plant. For example, if we're planting a carrot, we'll use a little carrot top that's been saved from another grown-up carrot!\n3. Then, we put the seed or root in the ground. We make sure it's nice and cozy down there.\n4. After that, we cover the seed or root with some special dirt called soil. This helps keep everything safe and comfy for our little plant to grow.\n5. Finally, w

# LCEL: LangChain Expression Langage
On the one hand ,it allows us to create some AI pipeline. On the other hand it is clean ,composable and declarative.In fact , we have just to chain our instruction.
Pro:
* Readability
* Performance
* Flexibility
* Simplicity 

## Output parser
It sets the format of output. It is like , we say to our model: Dude , you must reply in this format. I dont want you behave as you want? When you will reply, speak as that.. Among them , we have
* JsonOutputParser
* CvOutputParser
* StrOutputParser
* Pydantic

In [21]:
llm = ChatOllama(model='mistral:latest')

In [32]:
parser = StrOutputParser()
chain = zero_shot_pompt | llm | parser

In [33]:
responseChain = chain.invoke({'concept':'engrais'})

In [34]:
print(responseChain)

 Hello there! So, you know how plants need food to grow big and strong, just like us? Well, "engrais" is what we call the special kind of food for plants! It's made from natural things like cow poop, bird droppings, or even rocks that have nutrients in them. When we sprinkle this magic powder around our plants, it helps them grow bigger and greener, just like when we eat healthy foods to help us grow tall and strong! Isn't that amazing?


# Lets see another sample
Here , we assign a role to our model and implement a prompt template for user query

In [35]:
prompt = ChatPromptTemplate.from_messages([
    ("system","you are an expert in agriculture"),
    ("user","respond to this question:{question} like you explain to a {user_type}. If you dont know , say it instead of giving falsy response")
])

chain2 = prompt | llm | parser 

input_ = {"question":"how to increase his productivity","user_type":"kid"}

response2 = chain2.invoke(input_)

In [36]:
response2

" Sure! Let's imagine we have a magic garden where everything grows really fast, just like in fairytales. But since we don't have magic in the real world, I will tell you how to help your plants grow and produce more like a superhero gardener:\n\n1. Give plants their favorite snack: Plants love food! We can feed them with nutrient-rich compost or fertilizer. Remember not to overfeed them, as too much food might make them sick.\n\n2. Watering time! Just like you, plants need water to stay healthy and grow. Make sure they get enough water, but don't drown them – they can get sick if their roots are always wet. A good rule of thumb is to water the soil around the base of the plant, about 1 inch per week for most plants.\n\n3. Sunshine time: Most plants need plenty of sunlight to grow. Try to find a sunny spot in your garden for them. Remember that some plants like shade and don't like too much sun, so always check what kind of plants you have before you plant them!\n\n4. Give them space t

# Jsonoutputparser
That way the ouput will be in a JSON format.

In [37]:
# Create your JSON parser
json_parser = JsonOutputParser()

# Create more explicit format instructions
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON.
"""

# Create a prompt template with clearer instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain without cleaning step
movie_chain = prompt_template | llm | json_parser

# Test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")


Parsed result:
Title: The Matrix
Director: Lana Wachowski, Lilly Wachowski
Year: 1999
Genre: Science Fiction, Action


#### We can also see the output format
It is in Json format

In [39]:
result

{'title': 'The Matrix',
 'director': 'Lana Wachowski, Lilly Wachowski',
 'year': 1999,
 'genre': 'Science Fiction, Action'}

# Document
Langchain permits us to handle documents. It contains 2 part:
* page_content: We put there the document text
* Metadata: It contains some informations about the document. It can be id, author ...It is a Optional params,that means it is not obliged to set this params

# Documents loader
They aimed at load documents . It can be pdf , html

In [40]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader ,WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


### Lets create a document

In [42]:
create_document  = Document(
    page_content='Math is wonderful',
    metadata={
        "document_id":"99",
        "document_author":"Paul yves etiens"
    }
)
create_document

Document(metadata={'document_id': '99', 'document_author': 'Paul yves etiens'}, page_content='Math is wonderful')

## Loading a document

In [44]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

load_document = loader.load()
print(doc)

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental di

In [45]:
print(load_document[2]) # That way we can see the second page

page_content='Figure 2. An AIMessage illustration 
C. Prompt Template 
Prompt templates [10] allow you to structure input for LLMs. 
They provide a convenient way to format user inputs and 
provide instructions to generate responses. Prompt templates 
help ensure that the LLM understands the desired context and 
produces relevant outputs. 
The prompt template classes in LangChain are built to 
make constructing prompts with dynamic inputs easier. Of 
these classes, the simplest is the PromptTemplate. 
D. Chain 
Chains [11] in LangChain refer to the combination of 
multiple components to achieve specific tasks. They provide 
a structured and modular approach to building language 
model applications. By combining different components, you 
can create chains that address various u se cases and 
requirements. Here are some advantages of using chains: 
• Modularity: Chains allow you to break down 
complex tasks into smaller, manageable 
components. Each component can be developed and 
teste

In [161]:
print(doc[3].page_content[500:]) 

atbot in providing 
mental health support. It assures users of a safe and 
confidential space to express their concerns.  
Step 2. User Input - Prompt: Users can input mental health-
related questions or seek advice by typing their queries 
into the input box integrated into the Streamlit interface. 
Step 3. Data Transfer to LangChain: Implement the 
functionality that sends the user's input (question) as a 
chat prompt template to the LangChain framework. This 
input serves as the "human message prompt" template. 
Step 4. LangChain Framework: In this phase, the LangChain 
framework serves as the backbone of the chatbot, where 
all the foundational components and building blocks are 
meticulously orchestrated. Here's a deeper dive into the 
critical elements of LangChain Processing: 
• ChatMessage and Prompt Templates:  Within 
LangChain, the chatbot's core communication 
infrastructure is established by  creating 
ChatMessage and prompt templates for optimal 
chatbot engagement. 
• LL

# Loading a website content

In [46]:
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

web_data = loader.load()

print(web_data[0].page_content[:700])

LangChain overview - Docs by LangChainSkip to main contentðŸš€ Share how you're building agents for a chance to win LangChain swag!Docs by LangChain home pageLangChain + LangGraphSearch...âŒ˜KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep AgentsIntegrationsLearnReferenceContributePythonOverviewChangelogGet startedInstallQuickstartPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewBuilt-in middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAg


# Chunking
 We split can texts or phrase in smaller part called chunck. It is a useful because this chunks will be used to create embeddings

In [47]:
# chunk_overlap=20, That way, we keep context
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

chunks = text_splitter.split_documents(doc)

print('Chunk length',len(chunks))
print(chunks[100].page_content)

Chunk length 147
challenges or concerns you may have. Please feel free 
to share what's on your mind, and we'll work together 
to address your needs. Remember, this is a safe and


# Loading a document and splitting it

In [48]:

# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = CharacterTextSplitter(chunk_size=400,chunk_overlap=20,separator="\n")

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")


=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.80 characters
Metadata keys preserved: total_pages, creationdate, producer, page, author, moddate, title, page_label, source, creator

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Min chunk size: 49 characters
Max chunk size: 299 characters

=== Splitter 2 Statistics ===
Total number of chunks: 71
Average chunk size: 353.62 characters
Metadata keys preserved: total_pages, creationdate, producer, page, aut

# Chroma and Embedding
Embedding: It is vector which represent a token or word. LLM doesnt understand our langage. We need to transform words in a format they can understand. 
It is for this reason we use vectors which are a numerical representation of words for instance.

Chroma: It is a vector database. It enables us to store our embeddings

In [ ]:
embedding = OllamaEmbeddings(model="mistral:latest")

docsearch = Chroma.from_documents(chunks_1,embedding)

## Retrieve
We convert token in vector.That way , we are able to compare the similarity between this token

In [ ]:
query = 'langchain'

similar_docs = docsearch.similarity_search(QUERY)
print(similar_docs.page_content[0])

##### **Vector store-backed retrievers**


In [ ]:
retriever = docsearch.as_retriever()

query = 'rag and langchain'

retrieve_documents = retriever.invoke(query)

In [ ]:
retrieve_documents